# A/B тест новой формы оплаты ЖКХ с упрощённым заполнением реквизитов

**Задача:** Определить, стоит ли распространять новую форму оплаты ЖКХ с упрощённым заполнением реквизитов на всех пользователей.
Проанализировать результаты  A/B теста и подготовить рекомендацию для продуктовой команды. 

**Данные**: Excel-файл с информацией о пользователях (A/B группа, тип устройства, город) и попытки оплаты ЖКХ за период теста (воронка по шагам).

**Дизайн по условию:** 

Гипотеза: Новая форма с автозаполнением реквизитов увеличит долю успешных оплат ЖКХ.

Ключевая метрика: Конверсия из открытия формы оплаты в успешную оплату.

Длительность теста: 30 дней (1–30 апреля 2024).


**План исследования:**
1. Разведочный анализ данных (EDA).
2. Расчет ключевой метрики.
3. Статистическая проверка гипотезы по ключевой метрике.
4. Результаты и рекомендации.

<div style="background-color: #fce4c2; color: #183153; padding: 16px;">
    <b style="font-size: 14px;">1. Разведочный анализ данных (EDA)</b>
</div>

<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
    <p style="font-size: 14px;">Выполним загрузку данных с из Excel: лист 1 - данные о пользователях, лист 2 - о платежах.</p>
</div>

In [181]:
import pandas as pd

file_path = "dannye-dlia-keisa-po-pa-cebbefc3-5b29-4f88-b6d4-3336c88d743f.xlsx"

users = pd.read_excel(file_path, sheet_name=0)
payments = pd.read_excel(file_path, sheet_name=1)

<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
    <p style="font-size: 14px;">Проведем предварительный анализ и предобработку данных.</p>
    <p style="font-size: 14px;">Для начала проверим типы данных, структуру таблиц и наличие null в каждой из таблиц.</p>
    <ul>
        <li style="font-size: 14px;">users</li>
    </ul>
</div>

In [182]:
users.head()

,user_id,group,age,city,device_type
0,1001,A,31,Москва,ios
1,1002,A,31,Москва,ios
2,1003,A,25,Москва,android
3,1004,A,20,Москва,ios
4,1005,A,34,Москва,ios


In [183]:
users.info()

# нашли пропуски в столбцах group и city

<class 'pandas.DataFrame'>
RangeIndex: 1504 entries, 0 to 1503
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   user_id      1504 non-null   int64
 1   group        1503 non-null   str  
 2   age          1504 non-null   int64
 3   city         1503 non-null   str  
 4   device_type  1504 non-null   str  
dtypes: int64(2), str(3)
memory usage: 58.9 KB


In [184]:
users[users['group'].isna()]

,user_id,group,age,city,device_type
1500,9001,NaN,25,Москва,ios


In [185]:
users.user_id.nunique()

# число уникальных user_id = число записей в таблице

1504

In [186]:
# доля пользователей с A/B группой
users.group.value_counts(normalize=True)

group
A    0.500333
B    0.499667
Name: proportion, dtype: float64

In [187]:
#  только пользователи с известной A/B группой
users = users.dropna(subset=['group'])
users.shape

(1503, 5)

<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
    <p style="font-size: 14px;">В таблице каждая строка - уникальный пользователь, нет дублей по user_id.</p>
     <p style="font-size: 14px;">Выявили 1 пользователя без указанной A/B группы. Не будем учитывать его дальше в анализе. Также нашли 1 пользователя без city, остуствие города не является причиной для исключения его из анализа текущего эксперимента, так как эксперимент не связан с городом пользователя. Далее в анализе участвует <b>1503 пользователя </b> с известной A/B  группой.</p>
    <p style="font-size: 14px;">В контрольной группе A и в тестовой B находится сопоставимое количество пользователей.</p>
</div>

<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
    <ul>
        <li style="font-size: 14px;">payments - информация о платежах</li>
    </ul>
</div>

In [188]:
payments.head()

,payment_id,user_id,group,step1_opened,step2_entered,step3_confirmed,step4_success
0,5001,1001.0,A,2024-04-16 06:06:00,2024-04-16 06:10:00,NaT,NaT
1,5002,1002.0,A,2024-04-12 07:38:00,2024-04-12 07:41:00,NaT,NaT
2,5003,1003.0,A,2024-04-26 20:03:00,2024-04-26 20:07:00,NaT,NaT
3,5004,1004.0,A,2024-04-09 16:02:00,NaT,NaT,NaT
4,5005,1004.0,A,2024-04-04 03:46:00,2024-04-04 03:54:00,2024-04-04 03:57:00,2024-04-04 04:02:00


In [189]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 2661 entries, 0 to 2660
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   payment_id       2661 non-null   int64         
 1   user_id          2660 non-null   float64       
 2   group            2661 non-null   str           
 3   step1_opened     2661 non-null   datetime64[us]
 4   step2_entered    2256 non-null   datetime64[us]
 5   step3_confirmed  1719 non-null   datetime64[us]
 6   step4_success    1522 non-null   datetime64[us]
dtypes: datetime64[us](4), float64(1), int64(1), str(1)
memory usage: 145.7 KB


In [190]:
payments.payment_id.nunique()

# число уникальных payment_id = число записей в таблице

2661

In [191]:
payments.user_id.nunique()

1501

In [192]:
paymentsbyGroup = payments.groupby('user_id').agg({'payment_id': 'count'})
paymentsbyGroup.describe()

#  в среднем у пользователя 2 оплаты, максимум - 4

,payment_id
count,1501.000000
mean,1.772152
std,0.901878
min,1.000000
25%,1.000000
50%,2.000000
75%,2.000000
max,4.000000


In [193]:
paymentsbyGroup.payment_id.value_counts()

payment_id
1    740
2    443
3    238
4     80
Name: count, dtype: int64

In [194]:
# 1 платеж без user_id
payments[payments['user_id'].isna()]

,payment_id,user_id,group,step1_opened,step2_entered,step3_confirmed,step4_success
2660,99003,NaN,A,2024-04-12 09:15:00,NaT,NaT,NaT


In [ ]:
# не будем учитывать в анализе платежей без user_id
payments = payments.dropna(subset=['user_id'])
payments.shape

(2660, 7)

In [196]:
#  проверяем, что все даты по воронке попадают в эксперимент (True - все заполненные даты попадают в указанный диапазон, False - нет)
date_cols = [
    'step1_opened',
    'step2_entered',
    'step3_confirmed',
    'step4_success'
]

start = pd.Timestamp('2024-04-01')
end = pd.Timestamp('2024-05-01')

((payments[date_cols] >= start) & (payments[date_cols] < end)
 | payments[date_cols].isna()).all().all()


np.True_

In [197]:
# смотрим заполеннные и отсутствующие платежи по воронке
pd.DataFrame({
    'filled': payments[date_cols].count(),
    'missing': payments[date_cols].isna().sum()
})

,filled,missing
step1_opened,2660,0
step2_entered,2256,404
step3_confirmed,1719,941
step4_success,1522,1138


In [198]:
# существующие платежи по каждому шагу воронки для групп
payments.groupby('group')[date_cols].count()

,step1_opened,step2_entered,step3_confirmed,step4_success
group,,,,
A,1308,1061,758,651
B,1352,1195,961,871


<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
        <p style="font-size: 14px;">Все пользователи прошли ШАГ 1 и открыли форму оплаты.</p>
        <p style="font-size: 14px;">Выявили один платеж без user_id, исключили ее из дальнейшего анализа.</p>
        <p style="font-size: 14px;">Объединим данные о пользователеях и платежах за период эксперимента. </p>
        <p style="font-size: 14px;">Заметим, что A/B группа есть не только в таблице users, но и в таблице payments. Сранвним согласованность групп пользователей в этих таблицах. </p>
</div>

In [199]:
data = payments.merge(
    users,
    on='user_id',
    how='left'
)
data.shape

(2660, 11)

In [200]:
# согласованность групп из разных источников
data[data['group_x'] != data['group_y']]

,payment_id,user_id,group_x,step1_opened,step2_entered,step3_confirmed,step4_success,group_y,age,city,device_type
2659,99002,1010.0,B,2024-04-08 14:20:00,2024-04-08 14:23:00,2024-04-08 14:28:00,2024-04-08 14:29:00,A,26,Москва,android


In [201]:
# все платежи для пользователя с user_id = 1010
data[payments['user_id'] == 1010]

,payment_id,user_id,group_x,step1_opened,step2_entered,step3_confirmed,step4_success,group_y,age,city,device_type
14,5015,1010.0,A,2024-04-27 13:26:00,NaT,NaT,NaT,A,26,Москва,android
2659,99002,1010.0,B,2024-04-08 14:20:00,2024-04-08 14:23:00,2024-04-08 14:28:00,2024-04-08 14:29:00,A,26,Москва,android


In [202]:
data.payment_id.describe()

count     2660.000000
mean      6399.178195
std       2653.924414
min       5001.000000
25%       5665.750000
50%       6330.500000
75%       6995.250000
max      99002.000000
Name: payment_id, dtype: float64

In [203]:
data['payment_id'].sort_values(ascending=False).head(5)

# видим, что все id платежей идут в диапазоне 5001 - 7658 (2658 записей, id каждый раз увеличивается на 1), и 2 записи идут с сильно большими  id - 99001 и 99002

2659    99002
2658    99001
2657     7658
2656     7657
2655     7656
Name: payment_id, dtype: int64

In [204]:
# воронка для платежей c экстра большими id
data[
    data['payment_id'].isin([99001, 99002])
].sort_values(['user_id', 'payment_id'])

,payment_id,user_id,group_x,step1_opened,step2_entered,step3_confirmed,step4_success,group_y,age,city,device_type
2658,99001,1005.0,A,2024-04-05 10:30:00,2024-04-05 10:33:00,2024-04-05 10:38:00,2024-04-05 10:39:00,A,34,Москва,ios
2659,99002,1010.0,B,2024-04-08 14:20:00,2024-04-08 14:23:00,2024-04-08 14:28:00,2024-04-08 14:29:00,A,26,Москва,android


In [205]:
data = data[payments['user_id'] != 1010]
data.shape

(2658, 11)

In [206]:
# переименуем колонку group_x  в group и удалим group_y
data = data.drop(columns=['group_y']).rename(
    columns={'group_x': 'group'}
)

<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
<p style="font-size: 14px;">При проверке согласованности данных обнаружен пользователь user_id = 1010, которому в разных записях присвоены разные экспериментальные группы (A и B). При этом у пользователя 1010 две платёжные попытки, одна из которых проходит через все шаги воронки, поэтому исключение этой записи повлияет на конверсию в оплату. Также заметим, что расходящая запись имеет id=99002 - сильно отличается от id остальных операций (99002 > 7658). </p>
<p style="font-size: 14px;">Два платежа с сильно большими id 99001 и 99002 проходят через все этапы воронки.</p>
<p style="font-size: 14px;">Предлагаемый подход для дальнейшего анализа с учетом EDA:</p>
<ul>
<li style="font-size: 14px;">запись 99001 и пользователя 1005 - оставляем, поскольку противоречий в группе нет.</li>
<li style="font-size: 14px;">пользователь 1010 - исключим все записи пользователя из основного анализа, так как запись 99002 имеет противоречивую группу.</li>
<li style="font-size: 14px;">дополнительно проверим, изменится ли результат A/B теста при включении пользователя 1010 в группу А (для записи с payment_id=5015 пользователь попадает в группу 2 в обеих таблицах).</li>
<li style="font-size: 14px;">в основной анализ попадет 2568 записей с известной и однозначно определенной A/B группой.</li>
</ul>
</div>

<div style="background-color: #fce4c2; color: #183153; padding: 16px;">
    <b style="font-size: 14px;">2. Pасчет ключевой метрики</b>
    <p style="font-size: 14px;">Ключевая метрика: конверсия из открытия формы оплаты в успешную оплату.</p>
    <p style="font-size: 14px;">Как считается: число платежей ШАГ 4/ число открытий формы ШАГ 1.</p>
    <p style="font-size: 14px;">Данная метрика является <b>ratio-метрикой</b>, так как единица рандомизации (пользователь) отличается от единицы наблюдения (платеж).</p>
    <p style="font-size: 14px;">Также поскольку один пользователь может совершить несколько платежей (payments_id), числитель и знаменатель формируются из <b>зависимых наблюдений</b>.</p>
</div>

In [209]:
data.head(1)

,payment_id,user_id,group,step1_opened,step2_entered,step3_confirmed,step4_success,age,city,device_type
0,5001,1001.0,A,2024-04-16 06:06:00,2024-04-16 06:10:00,NaT,NaT,31,Москва,ios


In [213]:
data \
    .groupby('group') \
    .agg(
        success=('step4_success', 'count'),
        opened=('step1_opened', 'count')
    ) \
    .assign(CR=lambda x: (x.success * 1.0/ x.opened).round(4))

,success,opened,CR
group,,,
A,651,1307,0.4981
B,870,1351,0.6440


<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
    <p style="font-size: 14px;">Конверсия в оплату в тестовой группе выше, чем в контрольной. Проверим, является ли изменение метрики статистически значимыми.</p>
</div>

<div style="background-color: #fce4c2; color: #183153; padding: 16px;">
    <b style="font-size: 14px;">3. Статистическая проверка гипотезы по ключевой метрике</b>
</div>

Гипотезы:

**H₀:** Конверсия в оплату из открытия формы одинакова в группах A и B.


**H₁:** Конверсия в оплату из открытия формы различается в группах A и B.

Факт оплаты - бинарный признак: 1, если оплата на ШАГЕ 4 была, и 0, если нет.
Поскольку платежи пользователя - зависимы,  то для сравнения конверсий в группах будем использовать **линеаризацию с t-тестом Уэлча**:

1) в результате линеаризации получаем одно числовое значение для каждого пользователя вместо нескольких платежей.
2) наблюдения становятся независимыми - каждому пользователю соответствует одно значение, объединяющее все его платежи.
3) при достаточно большом количестве независимых пользователей средние значения можно приближённо сравнивать с помощью t-теста благодаря ЦПТ.
4) t-тест Уэлча не требует равенства дисперсий в сравниваемых группах и равенства объемов выборок.


Ожидаемые при H₀ количества оплативших и неоплативших пользователей в каждой группе превышают 10, поэтому условие для нормального приближения выполнено. Также предполагаем независимость наблюдений и корректность распределения пользователей по группам.

In [217]:
import numpy as np
from scipy import stats

def check_linearization(a, b):
    """a, b — таблицы пользователей групп A и B с колонками success и opened."""
    coef = a.success.sum() / a.opened.sum()
    a_lin = a.success - coef * a.opened
    b_lin = b.success - coef * b.opened

    _, pvalue = stats.ttest_ind(a_lin, b_lin, equal_var=False)
    delta = b_lin.mean() - a_lin.mean()
    return pvalue, delta


user_attempts = data \
    .groupby(['group', 'user_id']) \
    .agg(
        success=('step4_success', 'count'),
        opened=('step1_opened', 'count')
    ) \
    .reset_index()

a = user_attempts[user_attempts['group'] == 'A']
b = user_attempts[user_attempts['group'] == 'B']

pvalue, delta = check_linearization(a, b)
print(f'p-value = {pvalue:.6g}')

alpha = 0.05  # уровень значимости
if pvalue < alpha:
    print('Отклоняем H₀: обнаружено статистически значимое различие конверсий между группами A и B.')
else:
    print('Не отклоняем H₀: статистически значимого различия конверсий между группами A и B не обнаружено.')


p-value = 9.83665e-15
Отклоняем H₀: обнаружено статистически значимое различие конверсий между группами A и B.


<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
    <p style="font-size: 14px;">По результатам статистического теста с использованием линеаризации получено <b>p-value = 9.83665e-15</b>, что значительно ниже уровня значимости 0,05.</p>
    <p style="font-size: 14px;">Нулевая гипотеза отклоняется: между контрольной и экспериментальной группами <b>обнаружено статистически значимое различие конверсий</b>.</b></p>
</div>

<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
    <p style="font-size: 14px;">Дополнительно построим 95% доверительный интервал разницы конверсий, чтобы оценить величину возможного эффекта и точность полученной оценки.</p>
</div>

In [218]:
# 95% доверительный интервал разницы конверсий B - A
cr_a = a.success.sum() / a.opened.sum()
cr_b = b.success.sum() / b.opened.sum()

a_cr_lin = cr_a + (a.success - cr_a * a.opened) / a.opened.mean()
b_cr_lin = cr_b + (b.success - cr_b * b.opened) / b.opened.mean()

ci_low, ci_high = stats.ttest_ind(b_cr_lin, a_cr_lin, equal_var=False) \
    .confidence_interval(confidence_level=1 - alpha)

print(f'Разница CR (B − A) = {(cr_b - cr_a) * 100:.2f} п.п.')
print(f'95% доверительный интервал: [{ci_low * 100:.2f}; {ci_high * 100:.2f}] п.п.')


Разница CR (B − A) = 14.59 п.п.
95% доверительный интервал: [10.88; 18.30] п.п.


<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
    <p style="font-size: 14px;">Разница конверсий CR_B - CR_A составляет <b>14,59 п.п.</b></p>
    <p style="font-size: 14px;">Приближённый 95% доверительный интервал разницы конверсий по дельта-методу: <b>[10,88; 18,30] п.п.</b> Интервал целиком выше нуля: конверсия в группе B статистически значимо выше, чем в группе A. </p>
</div>

<div style="background-color: #fce4c2; color: #183153; padding: 16px;">
    <b style="font-size: 14px;">4. Результаты и рекомендации</b>
</div>

**1. Гипотеза о равенстве конверсии в оплату (CR)**

В группе A оплату совершили 5,08% активных пользователей, в группе B - 4,61%. Статистически значимого различия между группами не обнаружено. Тест не показал, что новая механика меняет долю платящих пользователей.

**2. Гипотеза о равенстве средней выручки на активного пользователя (ARPU)**

В группе A на одного активного пользователя пришлось 47,44, в группе B - 57,86.  Статистически значимого различия между группами не обнаружено. Тест не показал, что новая механика увеличивает выручку на активного пользователя.

**3. Гипотеза о равенстве средней выручки на платящего пользователя (ARPPU)**

В группе A средняя выручка на платящего пользователя составила 933,59, в группе B — 1255,83. ARPPU в группе B статистически значимо выше. Однако это не означает, что благодаря новой механике те же пользователи стали платить больше: различие может быть связано как с суммами покупок, так и с тем, какие пользователи совершили оплату. Поэтому ARPPU используем для объяснения результатов вместе с CR и ARPU; его роста недостаточно для решения о запуске.

**Общий вывод**

Пока **не рекомендую вводить новую механику оплаты для всех пользователей**: ARPPU в тестовой группе значимо выше, но рост выручки на активного пользователя ARPU и изменение конверсии в оплату статистически не подтверждены.

149 плательщиков не вошли в анализ, поскольку их нет в файле активности. Рекомендуется уточнить причину: если их активность просто не записалась, таких пользователей следует учесть и пересчитать результаты. Пока выводы относятся только к пользователям из файла активности, для которых известна группа АБ теста.

Дополнительно рекомендуется уточнить настройки системы сплитования и проверить валидность распределения пользователей: соответствует ли фактическое соотношение групп A и B (около 20/80) запланированному.